In [ ]:
import ee
ee.Initialize(project="sentinel-487715")

import geopandas as gpd

roads = gpd.read_file("/Users/miranda/Documents/GitHub/Sentinel-FYP/data/ghana-260216-free.shp/gis_osm_roads_free_1.shp")
roads = roads.to_crs("EPSG:4326")
roads.head()

from shapely.geometry import box

aoi = box(-0.6, 5.3, 0.2, 5.9)  # Accra bbox
roads_aoi = roads[roads.intersects(aoi)].copy()
len(roads_aoi)

In [ ]:
import datetime as dt

ghana_bbox = ee.Geometry.Rectangle([-0.6, 5.3, 0.2, 5.9])
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(ghana_bbox)
      .filterDate("2023-01-01", "2023-12-31")
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
      .select(["B2","B3","B4","B8","B11","B12"]))

composite = s2.median()

from shapely.geometry import LineString
import pandas as pd

def midpoint_of_linestring(ls: LineString):
    return ls.interpolate(0.5, normalized=True)

# take small sample for testing
sample = roads_aoi.sample(50, random_state=0).copy()
sample["midpt"] = sample.geometry.apply(midpoint_of_linestring)

# build EE FeatureCollection of points
features = []
for i, pt in enumerate(sample["midpt"]):
    features.append(ee.Feature(ee.Geometry.Point([pt.x, pt.y]), {"id": i}))
fc = ee.FeatureCollection(features)

# sample the composite at points
sampled = composite.sampleRegions(collection=fc, scale=10, geometries=True)
df = pd.DataFrame(sampled.getInfo()["features"])
# flatten properties
rows = [f["properties"] for f in sampled.getInfo()["features"]]
df = pd.DataFrame(rows)
df.head()


In [ ]:
import geemap

m = geemap.Map(center=[5.6, -0.2], zoom=10)
m.add_basemap("HYBRID")

# your Sentinel layer (assuming composite is defined)
m.addLayer(composite, {"bands":["B4","B3","B2"], "min":0, "max":3000}, "S2 True Color")

# add roads as vector overlay
m.add_gdf(roads_aoi, layer_name="OSM Roads (shapefile)")

m

In [ ]:
#import ee
#import geopandas as gpd
#import geemap
#import folium

#ee.Initialize(project="sentinel-487715")

# Load ALL Ghana roads
roads = gpd.read_file("/Users/miranda/Documents/GitHub/Sentinel-FYP/data/ghana-260216-free.shp/gis_osm_roads_free_1.shp")
roads = roads.to_crs("EPSG:4326")

# Optional: simplify to improve performance (adjust tolerance)
# roads["geometry"] = roads["geometry"].simplify(0.00015, preserve_topology=True)

# Sentinel-2 composite for Ghana bbox
ghana_bbox = ee.Geometry.Rectangle([-3.5, 4.5, 1.5, 11.5])
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(ghana_bbox)
      .filterDate("2023-01-01", "2023-12-31")
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
      .select(["B2","B3","B4","B8","B11","B12"]))
composite = s2.median()

m = geemap.Map(center=[7.95, -1.02], zoom=7)
m.add_basemap("HYBRID")
m.addLayer(composite, {"bands":["B4","B3","B2"], "min":0, "max":3000}, "S2 True Color")

# Bright road style
road_style = {
    "color": "#00FFFF",   # bright cyan
    "weight": 1.5,
    "opacity": 0.9
}

# Tooltip fields (adjust if needed)
tooltip_fields = ["osm_id", "code", "fclass", "name", "ref", "oneway", "maxspeed"]

# Add roads with hover tooltip
import geemap

m = geemap.Map(center=[7.95, -1.02], zoom=7)
m.add_basemap("HYBRID")

m.addLayer(composite, {"bands":["B4","B3","B2"], "min":0, "max":3000}, "S2 True Color")

# Keep only the fields you want to show on hover
tooltip_fields = ["osm_id", "code", "fclass", "name", "ref", "oneway", "maxspeed"]
roads_hover = roads[tooltip_fields + ["geometry"]].copy()

# Bright color + hover info
m.add_gdf(
    roads_hover,
    layer_name="OSM Roads (Ghana)",
    style={"color": "#00FFFF", "weight": 1.5, "opacity": 0.9},
    info_mode="on_hover"   # shows attributes on hover
)

m.add_layer_control()
m

